# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp /content/drive/MyDrive/data.zip /content/
# !unzip -q /content/data.zip -d /content/

Mounted at /content/drive


In [2]:
# 압축이 잘 풀렸는지 확인
!ls -l /content/

total 1821944
-rw------- 1 root root 1862691773 Aug 30 14:06 data.zip
drwxr-xr-x 2 root root     135168 Aug 28 09:44 dev
-rw-r--r-- 1 root root     707055 Aug 28 09:43 dev.csv
drwx------ 5 root root       4096 Aug 30 14:06 drive
drwxr-xr-x 1 root root       4096 Aug 24 13:28 sample_data
-rw-r--r-- 1 root root      81195 Aug 28 09:44 sample_submission.csv
drwxr-xr-x 2 root root     176128 Aug 28 09:45 test
-rw-r--r-- 1 root root     830494 Aug 28 09:44 test.csv
drwxr-xr-x 2 root root     176128 Aug 28 09:46 train
-rw-r--r-- 1 root root     852411 Aug 28 09:45 train.csv


In [3]:
import sys
print(sys.executable)

/usr/bin/python3


In [4]:
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [5]:
import sys

# 1. 기존 PyTorch 삭제 및 안정성이 검증된 버전으로 재설치
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128

# 2. 설치 확인
import torch
print("Torch version:", torch.__version__)
print("GPU Device:", torch.cuda.get_device_name())

# 3. 필수 라이브러리 설치 및 에러 유발 패키지(torchao) 삭제
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade
!pip uninstall -y -q torchao

Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.3 MB/s eta 0:00:00
Torch version: 2.11.0+cu128
GPU Device: NVIDIA A100-SXM4-80GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 140.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.1 MB/s eta 0:00:00
   ━━

In [6]:
import sys

# 1. 기존 PyTorch 삭제 및 안정성이 검증된 버전으로 재설치
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128

# 2. 설치 확인
import torch
print("Torch version:", torch.__version__)
print("GPU Device:", torch.cuda.get_device_name())

# # 3. 필수 라이브러리 설치 및 에러 유발 패키지(torchao) 삭제
# !pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade
# !pip uninstall -y -q torchao

# 3. 에러 유발 패키지 삭제 및 Qwen3.8 최신 아키텍처 지원 라이브러리 설치
!pip uninstall -y -q torchao

# 기존 버전을 설치하는 부분은 삭제하고, 곧바로 GitHub 최신 버전의 transformers와 Qwen 필수 패키지 설치
!pip install -q git+https://github.com/huggingface/transformers.git --upgrade
!pip install -q qwen-vl-utils "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl (

In [7]:
print('hello123')

hello123


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [1]:
import os, random, math
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from transformers import (
    # AutoModelForVision2Seq, # 8b
    AutoModelForImageTextToText, # 27b
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm



Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --- 최종 밤샘 구동 세팅 ---
# MODEL_ID = "Qwen/Qwen3.8-27B"  # 8B를 쓴다면 "Qwen/Qwen3-VL-8B-Instruct"로 변경
# MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 1024           # 고해상도 (VQA 성능 핵심)
MAX_NEW_TOKENS = 8
SEED = 42

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 전체 데이터 학습(5073개)을 진행하려면 아래 줄을 주석 처리하세요.
# 만약 1500장 서브셋 학습을 원하시면 주석을 해제하세요.
# train_df = train_df.sample(n=1500, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [2]:
# 양자화 - 27b
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*256,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "left"

# 사전학습 모델
# 7b 모델 기준
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="sdpa", # 🚨 OOM 방지 핵심: PyTorch 메모리 최적화 어텐션
    trust_remote_code=True,
)
# 8b + 양자화 X
# base_model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.bfloat16,
#     device_map="cuda",
#     attn_implementation="sdpa",
#     trust_remote_code=True,
# )


# 양자화 모델로 로드
base_model.config.use_cache = False
base_model.enable_input_require_grads()
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    # r=16,            # 27b
    r = 32,            # 8b
    # lora_alpha=32,   # 27b
    lora_alpha=64,     # 8b
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    # target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [3]:
# 모델 지시사항 (한국어)
SYSTEM_INSTRUCT = (
    "당신은 시각적 질의응답(VQA)을 완벽하게 수행하는 유능한 인공지능입니다. "
    "주어진 이미지와 질문을 분석한 뒤, 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 답변하세요."
)

# 프롬프트 (한국어)
def build_mc_prompt(question, a, b, c, d):
    return f"질문: {question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n정답:"

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [4]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        q = str(row["question"])
        options = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

        if self.train:
            ans_char = str(row["answer"]).strip().lower()
            ans_idx = ['a', 'b', 'c', 'd'].index(ans_char)
            correct_text = options[ans_idx]
            random.shuffle(options)
            final_answer = ['a', 'b', 'c', 'd'][options.index(correct_text)]
        else:
            final_answer = None

        user_text = build_mc_prompt(q, options[0], options[1], options[2], options[3])
        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
        ]

        if self.train:
            messages.append({"role":"assistant","content":[{"type":"text","text":final_answer}]})

        return {"messages": messages, "image": img, "prompt_only_messages": messages[:-1] if self.train else messages}

@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, prompt_texts, images = [], [], []
        for sample in batch:
            texts.append(self.processor.apply_chat_template(sample["messages"], tokenize=False, add_generation_prompt=False))
            prompt_texts.append(self.processor.apply_chat_template(sample["prompt_only_messages"], tokenize=False, add_generation_prompt=True))
            images.append(sample["image"])

        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        prompt_enc = self.processor(text=prompt_texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            labels = enc["input_ids"].clone()
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            for i in range(len(batch)):
                prompt_len = (prompt_enc["input_ids"][i] != self.processor.tokenizer.pad_token_id).sum()
                labels[i, :prompt_len] = -100
            enc["labels"] = labels
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [5]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)

train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)


# 데이터로더
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=0)
valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [6]:
model = model.to(device)
# GRAD_ACCUM = 16  # 27b, 유효 배치 사이즈 16 (안정적인 학습)
GRAD_ACCUM = 8  # 업데이트 주기를 적절히 당김

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 최신 문법의 scaler 적용
scaler = torch.amp.GradScaler('cuda', enabled=True)

for epoch in range(1): # 무조건 1에폭 고정
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            progress_bar.set_postfix({"loss": f"{running / GRAD_ACCUM:.3f}"})
            running = 0.0

    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen_2.5_7b"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


Epoch 1 [train]:   0%|          | 0/4565 [00:00<?, ?batch/s]

Epoch 1 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 1] valid loss 0.0808
Saved: /content/qwen_2.5_7b


In [ ]:
# import torch

# # 1. 알파벳별 실제 토큰 ID 확인
# check_tokens = ['a', 'b', 'c', 'd', ' a', ' b', ' c', ' d', ' A', ' B', ' C', ' D']
# print("🔍 [토크나이저 인코딩 매핑 확인]")
# for t in check_tokens:
#     t_id = processor.tokenizer.encode(t, add_special_tokens=False)
#     print(f"'{t}' -> Token ID: {t_id}")

# # 2. 모델이 실제로 무엇을 예측하는지 Top-5 확인 (test_df 0번째 샘플 기준)
# model.eval()
# sample_row = test_df.iloc[0]
# q = str(sample_row["question"])
# opts = [str(sample_row["a"]), str(sample_row["b"]), str(sample_row["c"]), str(sample_row["d"])]
# user_text = build_mc_prompt(q, opts[0], opts[1], opts[2], opts[3])

# messages = [
#     {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
#     {"role":"user","content":[{"type":"image","image": Image.open(sample_row["path"]).convert("RGB")},
#                               {"type":"text","text":user_text}]}
# ]

# text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = processor(text=[text_input], images=[Image.open(sample_row["path"]).convert("RGB")], padding=True, return_tensors="pt").to(device)

# with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
#     outputs = model(**inputs)

# # 프롬프트 직후(-1 위치) 모델이 예측한 Top 5 토큰 확인
# # ✅ 여기를 수정했습니다: next_token_logits를 float()으로 변환
# next_token_logits = outputs.logits[0, -1, :].float()
# top5_ids = torch.topk(next_token_logits, 5).indices
# top5_tokens = processor.tokenizer.convert_ids_to_tokens(top5_ids)
# top5_probs = torch.softmax(next_token_logits, dim=-1)[top5_ids].cpu().numpy()

# print(f"\n🚨 [프롬프트 마지막 토큰(-1 위치) 이후 Top-5 예측 결과]")
# for token, prob in zip(top5_tokens, top5_probs):
#     print(f"Token: '{token}' | Prob: {prob:.4f}")

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [7]:
model.eval()
candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

ROTATIONS = 4
BATCH_SIZE = 1     # 27B+1024px 모델은 추론도 batch 1이 안전합니다
# BATCH_SIZE = 4  # 8b

n_samples = len(test_df)

P_total = np.zeros((n_samples, 4))

for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples, BATCH_SIZE), desc=f"Inference [TTA Rot {rot}]", unit="batch"):
        chunk = test_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

            # 선지 위치 순환
            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total[s+k, orig_i] += lp[k, shown_i]

        # VRAM 메모리 파편화 방지
        del inputs, outputs
        torch.cuda.empty_cache()

# 평균 확률로 정답 도출
P_total /= ROTATIONS
final_preds = [candidate_tokens[idx] for idx in P_total.argmax(axis=1)]

# ✅ 파일명 수정 (2.5 7B 버전)
np.save("probability_2.5_all.npy", P_total)
print("확률값이 probability_2.5_all.npy 로 저장되었습니다.")

submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
submission.to_csv("/content/submission_2.5_all.csv", index=False)
print("성공적으로 저장되었습니다: /content/submission_2.5_all.csv")

Inference [TTA Rot 0]:   0%|          | 0/5074 [00:00<?, ?batch/s]

Inference [TTA Rot 1]:   0%|          | 0/5074 [00:00<?, ?batch/s]

Inference [TTA Rot 2]:   0%|          | 0/5074 [00:00<?, ?batch/s]

Inference [TTA Rot 3]:   0%|          | 0/5074 [00:00<?, ?batch/s]

확률값이 probability_2.5_all.npy 로 저장되었습니다.
성공적으로 저장되었습니다: /content/submission_2.5_all.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
import numpy as np

# 1. 대상 데이터를 test.csv가 아닌 dev.csv로 로드합니다.
dev_df = pd.read_csv("dev.csv")
n_samples_dev = len(dev_df)

P_total_dev = np.zeros((n_samples_dev, 4))
candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

ROTATIONS = 4
BATCH_SIZE = 1

print("🔥 Dev 데이터 추론 시작...")
for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples_dev, BATCH_SIZE), desc=f"Dev Inference [Rot {rot}]", unit="batch"):
        chunk = dev_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total_dev[s+k, orig_i] += lp[k, shown_i]

        del inputs, outputs
        torch.cuda.empty_cache()

P_total_dev /= ROTATIONS

# 2. dev 확률값 저장! (이게 있어야 라우팅 앙상블을 짤 수 있습니다)
np.save("dev_prob_2.5_all.npy", P_total_dev)
print("🎉 성공: dev_prob_2.5_all.npy 가 저장되었습니다.")

🔥 Dev 데이터 추론 시작...


Dev Inference [Rot 0]:   0%|          | 0/4413 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [9]:
import torch
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForMultimodalLM
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. 최신 메인 Expert 모델 로드
# ==========================================
# 27B-FP8을 쓰시려면 "Qwen/Qwen3.8-27B-FP8"
# 35B-A3B를 쓰시려면 "Qwen/Qwen3.6-35B-A3B" (현재 1픽 추천)
MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
IMAGE_SIZE = 1024 # Step 1: 512px / rot1 부터 시작

print(f"[{MODEL_ID}] 로딩 중...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*256,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True
)

# 🚨 핵심: 공식 가이드대로 MultimodalLM 클래스 및 dtype="auto" 사용
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    attn_implementation="sdpa",
    trust_remote_code=True
)
model.eval()
print("🎉 모델 로딩 완료!")

candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

# ==========================================
# 2. 프롬프트 및 TTA 설정
# ==========================================
SYSTEM_INSTRUCT = (
    "당신은 시각적 질의응답(VQA)을 완벽하게 수행하는 유능한 인공지능입니다. "
    "주어진 이미지와 질문을 분석한 뒤, 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 답변하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return f"질문: {question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n정답:"

ROTATIONS = 1
BATCH_SIZE = 1

# Validation을 위한 dev.csv 로드
target_df = pd.read_csv("dev.csv")
n_samples = len(target_df)
P_total = np.zeros((n_samples, 4))

# ==========================================
# 3. 제로샷 추론 (Thinking 모드 해제)
# ==========================================
for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples, BATCH_SIZE), desc=f"Inference [Rot {rot}]", unit="batch"):
        chunk = target_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]}
            ]

            # 🚨 3.8/3.6 등 최신 모델에서 바로 로짓을 뽑으려면 Thinking 모드 반드시 해제
            texts.append(processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            ))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to("cuda")

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total[s+k, orig_i] += lp[k, shown_i]

        del inputs, outputs
        if s % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

P_total /= ROTATIONS

# 모델명에 맞춰 저장
np.save(f"dev_prob_{MODEL_ID.split('/')[-1]}.npy", P_total)
print(f"✅ 성공: dev_prob_{MODEL_ID.split('/')[-1]}.npy 저장 완료!")

[Qwen/Qwen3.6-35B-A3B] 로딩 중...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/1026 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

🎉 모델 로딩 완료!


Inference [Rot 0]:   0%|          | 0/4413 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [1]:
import torch
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForMultimodalLM
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. 메인 Expert: Qwen3.5-27B (무손실 bfloat16 로드)
# ==========================================
MODEL_ID = "Qwen/Qwen3.5-27B"
IMAGE_SIZE = 1024

print(f"[{MODEL_ID}] 로딩 중...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*256,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True
)

# 🚨 양자화 없이 순정 bfloat16으로 통째로 적재
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    trust_remote_code=True
)
model.eval()
print("🎉 27B 순정 모델 로딩 완료!")

candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

# ==========================================
# 2. 프롬프트 및 TTA 설정
# ==========================================
SYSTEM_INSTRUCT = (
    "당신은 시각적 질의응답(VQA)을 완벽하게 수행하는 유능한 인공지능입니다. "
    "주어진 이미지와 질문을 분석한 뒤, 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 답변하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return f"질문: {question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n정답:"

ROTATIONS = 1
BATCH_SIZE = 1

# Validation을 위한 dev.csv 로드
# target_df = pd.read_csv("dev.csv")
target_df = pd.read_csv("test.csv")
n_samples = len(target_df)
P_total = np.zeros((n_samples, 4))

# ==========================================
# 3. 제로샷 추론 (Thinking 모드 해제)
# ==========================================
print(f"🔥 1024px 해상도로 추론 시작...")
for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples, BATCH_SIZE), desc=f"Inference [Rot {rot}]", unit="batch"):
        chunk = target_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]}
            ]

            # Qwen3 계열 호환을 위해 enable_thinking=False 적용
            texts.append(processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            ))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to("cuda")

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total[s+k, orig_i] += lp[k, shown_i]

        del inputs, outputs
        if s % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

P_total /= ROTATIONS

# np.save("dev_prob_qwen3.5_27b.npy", P_total)
np.save("probability_qwen3.5_27b.npy", P_total)
print("✅ 성공: dev_prob_qwen3.5_27b.npy 저장 완료!")

[Qwen/Qwen3.5-27B] 로딩 중...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

🎉 27B 순정 모델 로딩 완료!
🔥 1024px 해상도로 추론 시작...


Inference [Rot 0]:   0%|          | 0/5074 [00:00<?, ?batch/s]

✅ 성공: dev_prob_qwen3.5_27b.npy 저장 완료!


In [6]:
import torch
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel # 파인튜닝 가중치 로드용

# 🚨 에러 원인 해결: device 명시적 선언
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. 7B 파인튜닝 모델 및 프로세서 로드
# ==========================================
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
ADAPTER_PATH = "./qwen_2.5_7b" # 캡처해주신 파인튜닝 폴더 경로

print("모델 로딩 중... (Base + LoRA Adapter)")
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    min_pixels=256*256,
    max_pixels=1024*1024,
    trust_remote_code=True
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    trust_remote_code=True
)

# 베이스 모델에 파인튜닝 가중치(qwen_2.5_7b) 씌우기
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("🎉 7B 파인튜닝 모델 로드 완료!")

# 토큰 매핑
candidate_tokens = ['a', 'b', 'c', 'd']
candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

SYSTEM_INSTRUCT = (
    "당신은 시각적 질의응답(VQA)을 완벽하게 수행하는 유능한 인공지능입니다. "
    "주어진 이미지와 질문을 분석한 뒤, 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 답변하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return f"질문: {question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n정답:"

# ==========================================
# 2. Dev 데이터 추론 루프
# ==========================================
dev_df = pd.read_csv("dev.csv")
n_samples_dev = len(dev_df)

P_total_dev = np.zeros((n_samples_dev, 4))
ROTATIONS = 4
BATCH_SIZE = 1

print("🔥 Dev 데이터 추론 시작...")
for rot in range(ROTATIONS):
    for s in tqdm(range(0, n_samples_dev, BATCH_SIZE), desc=f"Dev Inference [Rot {rot}]", unit="batch"):
        chunk = dev_df.iloc[s:s+BATCH_SIZE]
        texts, images, perms = [], [], []

        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            q = str(row["question"])
            base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

            perm = [(i + rot) % 4 for i in range(4)]
            user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
            ]

            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)
            perms.append(perm)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].float()
            lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

        for k, perm in enumerate(perms):
            for shown_i, orig_i in enumerate(perm):
                P_total_dev[s+k, orig_i] += lp[k, shown_i]

        del inputs, outputs
        torch.cuda.empty_cache()

# ==========================================
# 3. 평균 확률 도출 및 저장
# ==========================================
P_total_dev /= ROTATIONS

np.save("dev_prob_2.5_all.npy", P_total_dev)
print("🎉 성공: dev_prob_2.5_all.npy 가 저장되었습니다.")

모델 로딩 중... (Base + LoRA Adapter)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

🎉 7B 파인튜닝 모델 로드 완료!
🔥 Dev 데이터 추론 시작...


Dev Inference [Rot 0]:   0%|          | 0/4413 [00:00<?, ?batch/s]

Dev Inference [Rot 1]:   0%|          | 0/4413 [00:00<?, ?batch/s]

Dev Inference [Rot 2]:   0%|          | 0/4413 [00:00<?, ?batch/s]

Dev Inference [Rot 3]:   0%|          | 0/4413 [00:00<?, ?batch/s]

🎉 성공: dev_prob_2.5_all.npy 가 저장되었습니다.


In [7]:
import numpy as np
import pandas as pd

CLASSES = np.array(["a", "b", "c", "d"])

# =========================================================
# 파일
# =========================================================

dev = pd.read_csv("dev.csv")

P = {
    "27B": np.load("dev_prob_qwen3.5_27b.npy"),
    "ALL": np.load("dev_prob_all.npy"),
    "1500": np.load("dev_prob_1500.npy"),
    "2.5B": np.load("dev_prob_2.5_all.npy"),
}

# =========================================================
# softmax
# =========================================================

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

P_soft = {
    k: softmax(v.astype(np.float64))
    for k, v in P.items()
}

# =========================================================
# majority GT
# =========================================================

from collections import Counter

answer_cols = [
    "answer1", "answer2", "answer3",
    "answer4", "answer5"
]

def majority(row):
    vals = [
        str(row[c]).strip().lower()
        for c in answer_cols
        if pd.notna(row[c])
        and str(row[c]).strip().lower() in CLASSES
    ]
    if not vals:
        return None
    return Counter(vals).most_common(1)[0][0]

gt = dev.apply(majority, axis=1).values
valid = pd.notna(gt)

# =========================================================
# single-model benchmark
# =========================================================

print("=" * 70)
print("SINGLE MODEL BENCHMARK")
print("=" * 70)

for name, p in P_soft.items():

    pred = p.argmax(axis=1)

    acc = np.mean(
        CLASSES[pred[valid]]
        ==
        gt[valid]
    )

    conf = p.max(axis=1)

    print(
        f"{name:6s} | "
        f"acc={acc:.4f} | "
        f"mean_conf={conf.mean():.4f}"
    )

SINGLE MODEL BENCHMARK
27B    | acc=0.4463 | mean_conf=0.7177
ALL    | acc=0.5150 | mean_conf=0.7030
1500   | acc=0.4796 | mean_conf=0.6755
2.5B   | acc=0.5054 | mean_conf=0.6898


In [8]:
import numpy as np
import pandas as pd
from collections import Counter

CLASSES = np.array(["a", "b", "c", "d"])
dev = pd.read_csv("dev.csv")

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

# 1. Dev 확률값 로드 및 Softmax 변환
P = {
    "ALL": softmax(np.load("dev_prob_all.npy").astype(np.float64)),
    "2.5B": softmax(np.load("dev_prob_2.5_all.npy").astype(np.float64)),
    "1500": softmax(np.load("dev_prob_1500.npy").astype(np.float64)),
    "27B": softmax(np.load("dev_prob_qwen3.5_27b.npy").astype(np.float64)),
}

answer_cols = ["answer1", "answer2", "answer3", "answer4", "answer5"]
def majority(row):
    vals = [str(row[c]).strip().lower() for c in answer_cols if pd.notna(row[c]) and str(row[c]).strip().lower() in CLASSES]
    return Counter(vals).most_common(1)[0][0] if vals else None

gt = dev.apply(majority, axis=1).values
valid = pd.notna(gt)
gt_valid = gt[valid]

P_all = P["ALL"][valid]
P_25b = P["2.5B"][valid]
P_1500 = P["1500"][valid]
P_27b = P["27B"][valid]

print("=" * 60)
print("🚀 STEP 3: Global Weight Search (Random Search 10,000 combos)")
print("=" * 60)

np.random.seed(42)
best_acc = 0
best_w = None

# 합이 1이 되는 4개 모델 가중치 조합 10,000개 무작위 생성
weights_pool = np.random.dirichlet((1, 1, 1, 1), 10000)

for w in weights_pool:
    p_blend = w[0]*P_all + w[1]*P_25b + w[2]*P_1500 + w[3]*P_27b
    acc = np.mean(CLASSES[p_blend.argmax(axis=1)] == gt_valid)
    if acc > best_acc:
        best_acc = acc
        best_w = w

print(f"✅ 최고 Soft Voting 정확도: {best_acc:.4f}")
print(f"✅ 최적 가중치 -> ALL: {best_w[0]:.3f}, 2.5B: {best_w[1]:.3f}, 1500: {best_w[2]:.3f}, 27B: {best_w[3]:.3f}")

# 최적 가중치로 생성된 기본 앙상블 베이스
P_blend_best = best_w[0]*P_all + best_w[1]*P_25b + best_w[2]*P_1500 + best_w[3]*P_27b
blend_preds = CLASSES[P_blend_best.argmax(axis=1)]

print("\n" + "=" * 60)
print("🎯 STEP 4: Confidence-Gated Routing Search")
print("=" * 60)

conf_all = P_all.max(axis=1)
conf_27b = P_27b.max(axis=1)
ans_27b = CLASSES[P_27b.argmax(axis=1)]

best_route_acc = best_acc
best_route_params = None

# 27B가 X% 이상 확신하고, ALL이 Y% 이하로 헷갈릴 때만 교체
expert_threshes = np.arange(0.60, 1.00, 0.05)
base_threshes = np.arange(0.30, 0.95, 0.05)

for exp_th in expert_threshes:
    for base_th in base_threshes:
        routed_preds = blend_preds.copy()
        route_mask = (conf_27b >= exp_th) & (conf_all <= base_th)
        routed_preds[route_mask] = ans_27b[route_mask]

        acc = np.mean(routed_preds == gt_valid)
        if acc > best_route_acc:
            best_route_acc = acc
            best_route_params = (exp_th, base_th, np.sum(route_mask))

if best_route_params:
    print(f"🔥 최고 라우팅 정확도: {best_route_acc:.4f} (Soft Voting 대비 추가 상승!)")
    print(f"🔥 황금 규칙: [27B 확신도 >= {best_route_params[0]:.2f}] AND [ALL 확신도 <= {best_route_params[1]:.2f}]")
    print(f"🛡️ 27B로 교체된 킬러 문항 수: {best_route_params[2]}개")
else:
    print("단순 가중치 앙상블(Step 3)을 이기는 라우팅 규칙이 발견되지 않았습니다. 27B의 과신 오답이 너무 많습니다.")

🚀 STEP 3: Global Weight Search (Random Search 10,000 combos)
✅ 최고 Soft Voting 정확도: 0.5254
✅ 최적 가중치 -> ALL: 0.593, 2.5B: 0.374, 1500: 0.024, 27B: 0.009

🎯 STEP 4: Confidence-Gated Routing Search
🔥 최고 라우팅 정확도: 0.5279 (Soft Voting 대비 추가 상승!)
🔥 황금 규칙: [27B 확신도 >= 0.85] AND [ALL 확신도 <= 0.70]
🛡️ 27B로 교체된 킬러 문항 수: 256개


In [11]:
import numpy as np
import pandas as pd

# 1. 클래스 정의 및 Softmax 함수
CLASSES = np.array(["a", "b", "c", "d"])

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

print("🚀 STEP 5: Test Set 추론 및 최종 앙상블 적용")

# 2. Test 확률값 로드 (미리 추출된 Test npy 파일이 있다고 가정)
# 만약 파일명이 다르다면 실제 Test npy 파일명으로 수정해주세요.
P_test = {
    "ALL": softmax(np.load("probability_all.npy").astype(np.float64)),
    "2.5B": softmax(np.load("probability_2.5_all.npy").astype(np.float64)),
    "1500": softmax(np.load("probability_1500.npy").astype(np.float64)),
    "27B": softmax(np.load("probability_qwen3.5_27b.npy").astype(np.float64)),
}

# 3. Dev에서 찾은 최적 가중치 적용
W_ALL, W_25B, W_1500, W_27B = 0.593, 0.374, 0.024, 0.009
P_test_blend = (W_ALL * P_test["ALL"] +
                W_25B * P_test["2.5B"] +
                W_1500 * P_test["1500"] +
                W_27B * P_test["27B"])

# 4. 베이스 앙상블 예측값 생성
final_preds = CLASSES[P_test_blend.argmax(axis=1)]

# 5. 황금 라우팅 규칙 적용
conf_test_all = P_test["ALL"].max(axis=1)
conf_test_27b = P_test["27B"].max(axis=1)
ans_test_27b = CLASSES[P_test["27B"].argmax(axis=1)]

# [27B 확신도 >= 0.85] AND [ALL 확신도 <= 0.70]
route_mask = (conf_test_27b >= 0.85) & (conf_test_all <= 0.70)
final_preds[route_mask] = ans_test_27b[route_mask]

print(f"🛡️ Test 데이터에서 27B로 교체된 킬러 문항 수: {np.sum(route_mask)}개")

# 6. 최종 제출 파일 생성 (sample_submission.csv가 있다고 가정)
sub = pd.read_csv("sample_submission.csv")
sub["answer"] = final_preds
sub.to_csv("final_submission_routed.csv", index=False)

print("🎉 성공: final_submission_routed.csv 파일이 생성되었습니다! 바로 제출해보세요.")

🚀 STEP 5: Test Set 추론 및 최종 앙상블 적용
🛡️ Test 데이터에서 27B로 교체된 킬러 문항 수: 158개
🎉 성공: final_submission_routed.csv 파일이 생성되었습니다! 바로 제출해보세요.


In [13]:
import numpy as np
import pandas as pd
from collections import Counter

CLASSES = np.array(["a", "b", "c", "d"])

# ============================================================
# LOAD
# ============================================================

dev = pd.read_csv("dev.csv")
test = pd.read_csv("test.csv")

dev_P = {
    "ALL": np.load("dev_prob_all.npy"),
    "25B": np.load("dev_prob_2.5_all.npy"),
    "1500": np.load("dev_prob_1500.npy"),
    "27B": np.load("dev_prob_qwen3.5_27b.npy"),
}

test_P = {
    "ALL": np.load("probability_all.npy"),
    "25B": np.load("probability_2.5_all.npy"),
    "1500": np.load("probability_1500.npy"),
    "27B": np.load("probability_qwen3.5_27b.npy"),
}


# ============================================================
# SOFTMAX
# ============================================================

def softmax(x):

    x = x.astype(np.float64)

    x = x - x.max(
        axis=1,
        keepdims=True
    )

    e = np.exp(x)

    return e / e.sum(
        axis=1,
        keepdims=True
    )


dev_prob = {
    k: softmax(v)
    for k, v in dev_P.items()
}

test_prob = {
    k: softmax(v)
    for k, v in test_P.items()
}


# ============================================================
# GT
# ============================================================

answer_cols = [
    "answer1",
    "answer2",
    "answer3",
    "answer4",
    "answer5"
]


def majority(row):

    vals = [
        str(row[c]).strip().lower()
        for c in answer_cols
        if pd.notna(row[c])
        and str(row[c]).strip().lower() in CLASSES
    ]

    if not vals:
        return None

    return Counter(vals).most_common(1)[0][0]


gt = dev.apply(
    majority,
    axis=1
).values

valid = pd.notna(gt)


# ============================================================
# SINGLE PRED
# ============================================================

dev_pred = {
    k: dev_prob[k].argmax(axis=1)
    for k in dev_prob
}

test_pred = {
    k: test_prob[k].argmax(axis=1)
    for k in test_prob
}


# ============================================================
# 1. HARD VOTE
# ============================================================

print("=" * 80)
print("HARD VOTE")
print("=" * 80)

model_names = [
    "ALL",
    "25B",
    "1500",
    "27B"
]


def hard_vote(pred_dict):

    out = np.zeros(
        len(next(iter(pred_dict.values()))),
        dtype=int
    )

    for i in range(len(out)):

        votes = [
            pred_dict[m][i]
            for m in model_names
        ]

        counts = Counter(votes)

        out[i] = counts.most_common(1)[0][0]

    return out


dev_vote = hard_vote(dev_pred)

vote_acc = np.mean(
    CLASSES[dev_vote[valid]]
    ==
    gt[valid]
)

print(
    f"Hard Vote Dev: "
    f"{vote_acc:.6f} "
    f"({vote_acc*100:.3f}%)"
)


# ============================================================
# 2. WEIGHTED HARD VOTE
# ============================================================

print("\n" + "=" * 80)
print("WEIGHTED HARD VOTE SEARCH")
print("=" * 80)

# weight는 "투표수" 개념으로 사용
weight_sets = [

    (5, 3, 1, 1),
    (5, 2, 1, 1),
    (6, 3, 1, 1),
    (6, 4, 1, 1),
    (7, 3, 1, 1),

    (5, 4, 1, 2),
    (6, 4, 1, 2),
    (7, 4, 1, 2),

    (6, 3, 2, 1),
    (7, 3, 2, 1),

    (6, 4, 2, 1),
    (7, 4, 2, 1),

    (5, 5, 1, 1),
]

vote_results = []

for w in weight_sets:

    score = np.zeros(
        (len(dev), 4)
    )

    for idx, model in enumerate(
        model_names
    ):

        for cls in range(4):

            score[:, cls] += (
                w[idx]
                *
                (dev_pred[model] == cls)
            )

    pred = score.argmax(
        axis=1
    )

    acc = np.mean(
        CLASSES[pred[valid]]
        ==
        gt[valid]
    )

    vote_results.append({
        "weights": w,
        "accuracy": acc
    })


vote_results = sorted(
    vote_results,
    key=lambda x: x["accuracy"],
    reverse=True
)

for x in vote_results[:10]:

    print(
        x["weights"],
        f"{x['accuracy']:.6f}"
    )


# ============================================================
# 3. VOTE + GOLDEN 27B RULE
# ============================================================

best_weight = vote_results[0]["weights"]

score = np.zeros(
    (len(dev), 4)
)

for idx, model in enumerate(
    model_names
):

    for cls in range(4):

        score[:, cls] += (
            best_weight[idx]
            *
            (dev_pred[model] == cls)
        )

vote_pred = score.argmax(
    axis=1
)


# 27B / ALL confidence
conf_all = dev_prob["ALL"].max(axis=1)
conf_27b = dev_prob["27B"].max(axis=1)

gold = (
    (conf_27b >= 0.85)
    &
    (conf_all <= 0.70)
    &
    (dev_pred["27B"] != dev_pred["ALL"])
)


for use_gold in [False, True]:

    final_pred = vote_pred.copy()

    if use_gold:

        final_pred[gold] = dev_pred["27B"][
            gold
        ]

    acc = np.mean(
        CLASSES[final_pred[valid]]
        ==
        gt[valid]
    )

    print(
        "\nVote + Gold:",
        use_gold,
        "changed:",
        gold.sum(),
        "acc:",
        acc
    )


# ============================================================
# 4. TEST HARD VOTE
# ============================================================

test_score = np.zeros(
    (len(test), 4)
)

for idx, model in enumerate(
    model_names
):

    for cls in range(4):

        test_score[:, cls] += (
            best_weight[idx]
            *
            (test_pred[model] == cls)
        )

test_vote_pred = test_score.argmax(
    axis=1
)


# ============================================================
# 5. TEST GOLDEN RULE
# ============================================================

test_conf_all = test_prob["ALL"].max(axis=1)
test_conf_27b = test_prob["27B"].max(axis=1)

test_gold = (
    (test_conf_27b >= 0.85)
    &
    (test_conf_all <= 0.70)
    &
    (test_pred["27B"] != test_pred["ALL"])
)

test_final = test_vote_pred.copy()

# gold rule 적용
test_final[test_gold] = test_pred["27B"][
    test_gold
]


# ============================================================
# 6. SAVE
# ============================================================

sub = pd.DataFrame({
    "id": test["id"],
    "answer": CLASSES[test_final]
})

sub.to_csv(
    "submission_hardvote_gold.csv",
    index=False
)

print("\n" + "=" * 80)
print("TEST")
print("=" * 80)

print(
    "Hard vote changed from All:",
    np.sum(
        test_vote_pred
        != test_pred["ALL"]
    )
)

print(
    "Gold routing:",
    test_gold.sum()
)

print(
    "Total changed from All:",
    np.sum(
        test_final
        != test_pred["ALL"]
    )
)

print(
    "Saved: submission_hardvote_gold.csv"
)

HARD VOTE
Hard Vote Dev: 0.509293 (50.929%)

WEIGHTED HARD VOTE SEARCH
(5, 2, 1, 1) 0.514959
(6, 3, 1, 1) 0.514959
(7, 3, 1, 1) 0.514959
(7, 3, 2, 1) 0.514959
(5, 3, 1, 1) 0.512693
(6, 4, 1, 1) 0.512693
(7, 4, 1, 2) 0.512693
(6, 3, 2, 1) 0.512693
(7, 4, 2, 1) 0.512693
(6, 4, 2, 1) 0.511786

Vote + Gold: False changed: 101 acc: 0.514959202175884

Vote + Gold: True changed: 101 acc: 0.5163191296464189

TEST
Hard vote changed from All: 0
Gold routing: 65
Total changed from All: 65
Saved: submission_hardvote_gold.csv


In [14]:
import numpy as np
import pandas as pd
from collections import Counter

# ============================================================
# PAIRWISE MODEL ANALYSIS
# ============================================================

CLASSES = np.array(["a", "b", "c", "d"])

dev = pd.read_csv("dev.csv")

P = {
    "ALL": np.load("dev_prob_all.npy"),
    "25B": np.load("dev_prob_2.5_all.npy"),
    "1500": np.load("dev_prob_1500.npy"),
    "27B": np.load("dev_prob_qwen3.5_27b.npy"),
}


def softmax(x):
    x = x.astype(np.float64)
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)


P = {
    k: softmax(v)
    for k, v in P.items()
}


# ============================================================
# GT
# ============================================================

answer_cols = [
    "answer1",
    "answer2",
    "answer3",
    "answer4",
    "answer5"
]


def majority(row):
    vals = [
        str(row[c]).strip().lower()
        for c in answer_cols
        if pd.notna(row[c])
        and str(row[c]).strip().lower() in CLASSES
    ]

    if not vals:
        return None

    return Counter(vals).most_common(1)[0][0]


gt = dev.apply(
    majority,
    axis=1
).values

valid = pd.notna(gt)


# ============================================================
# PREDICTION
# ============================================================

pred = {
    k: P[k].argmax(axis=1)
    for k in P
}


# ============================================================
# INDIVIDUAL
# ============================================================

print("=" * 80)
print("INDIVIDUAL")
print("=" * 80)

for k in pred:

    acc = np.mean(
        CLASSES[pred[k][valid]]
        ==
        gt[valid]
    )

    print(
        f"{k:6s}: {acc:.6f}"
    )


# ============================================================
# PAIRWISE
# ============================================================

pairs = [
    ("ALL", "25B"),
    ("ALL", "1500"),
    ("ALL", "27B"),
    ("25B", "1500"),
    ("25B", "27B"),
    ("1500", "27B"),
]


print("\n" + "=" * 80)
print("PAIRWISE ORACLE ANALYSIS")
print("=" * 80)

all_rows = []

for a, b in pairs:

    pa = pred[a]
    pb = pred[b]

    same = pa == pb
    disagree = ~same

    a_correct = (
        CLASSES[pa] == gt
    )

    b_correct = (
        CLASSES[pb] == gt
    )

    a_win = (
        disagree
        &
        a_correct
        &
        ~b_correct
        &
        valid
    )

    b_win = (
        disagree
        &
        ~a_correct
        &
        b_correct
        &
        valid
    )

    both_correct = (
        disagree
        &
        a_correct
        &
        b_correct
        &
        valid
    )

    both_wrong = (
        disagree
        &
        ~a_correct
        &
        ~b_correct
        &
        valid
    )

    n_dis = disagree[valid].sum()

    print(f"\n{a} vs {b}")
    print("-" * 50)
    print(
        "disagreement:",
        n_dis
    )
    print(
        f"{a} only correct:",
        a_win.sum()
    )
    print(
        f"{b} only correct:",
        b_win.sum()
    )
    print(
        "both correct:",
        both_correct.sum()
    )
    print(
        "both wrong:",
        both_wrong.sum()
    )

    if n_dis > 0:

        print(
            f"{a} win rate:",
            a_win.sum() / n_dis
        )

        print(
            f"{b} win rate:",
            b_win.sum() / n_dis
        )

    all_rows.append({
        "A": a,
        "B": b,
        "disagreement": n_dis,
        "A_only_correct": a_win.sum(),
        "B_only_correct": b_win.sum(),
        "both_correct": both_correct.sum(),
        "both_wrong": both_wrong.sum(),
    })


pair_df = pd.DataFrame(all_rows)

pair_df.to_csv(
    "pairwise_model_analysis.csv",
    index=False
)


# ============================================================
# ALL를 기준으로 누가 구해주는지
# ============================================================

print("\n" + "=" * 80)
print("WHO CAN RESCUE ALL?")
print("=" * 80)

base = pred["ALL"]

for helper in [
    "25B",
    "1500",
    "27B"
]:

    helper_pred = pred[helper]

    route = (
        (base != helper_pred)
        &
        valid
    )

    gt_v = gt[route]

    base_correct = (
        base[route]
        ==
        np.array(
            [np.where(CLASSES == x)[0][0] for x in gt_v]
        )
    )

    helper_correct = (
        helper_pred[route]
        ==
        np.array(
            [np.where(CLASSES == x)[0][0] for x in gt_v]
        )
    )

    rescue = (
        (~base_correct)
        &
        helper_correct
    )

    harm = (
        base_correct
        &
        (~helper_correct)
    )

    print(
        f"\nALL vs {helper}"
    )

    print(
        "disagreement:",
        route.sum()
    )

    print(
        "helper rescues ALL:",
        rescue.sum()
    )

    print(
        "helper harms ALL:",
        harm.sum()
    )

    print(
        "net:",
        rescue.sum() - harm.sum()
    )


print("\nSaved: pairwise_model_analysis.csv")

INDIVIDUAL
ALL   : 0.514959
25B   : 0.505440
1500  : 0.479601
27B   : 0.446283

PAIRWISE ORACLE ANALYSIS

ALL vs 25B
--------------------------------------------------
disagreement: 1365
ALL only correct: 533
25B only correct: 491
both correct: 0
both wrong: 341
ALL win rate: 0.3904761904761905
25B win rate: 0.3597069597069597

ALL vs 1500
--------------------------------------------------
disagreement: 1101
ALL only correct: 504
1500 only correct: 348
both correct: 0
both wrong: 249
ALL win rate: 0.45776566757493187
1500 win rate: 0.31607629427792916

ALL vs 27B
--------------------------------------------------
disagreement: 1830
ALL only correct: 854
27B only correct: 551
both correct: 0
both wrong: 425
ALL win rate: 0.4666666666666667
27B win rate: 0.3010928961748634

25B vs 1500
--------------------------------------------------
disagreement: 1518
25B only correct: 630
1500 only correct: 516
both correct: 0
both wrong: 372
25B win rate: 0.4150197628458498
1500 win rate: 0.33992094

In [15]:
import numpy as np
import pandas as pd
from collections import Counter

CLASSES = np.array(["a", "b", "c", "d"])

# ============================================================
# LOAD
# ============================================================

dev = pd.read_csv("dev.csv")

P = {
    "ALL": np.load("dev_prob_all.npy").astype(np.float64),
    "25B": np.load("dev_prob_2.5_all.npy").astype(np.float64),
    "27B": np.load("dev_prob_qwen3.5_27b.npy").astype(np.float64),
}

# ============================================================
# SOFTMAX
# ============================================================

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

P = {
    k: softmax(v)
    for k, v in P.items()
}

# ============================================================
# GT
# ============================================================

answer_cols = [
    "answer1",
    "answer2",
    "answer3",
    "answer4",
    "answer5"
]

def majority(row):
    vals = [
        str(row[c]).strip().lower()
        for c in answer_cols
        if pd.notna(row[c])
        and str(row[c]).strip().lower() in CLASSES
    ]

    if not vals:
        return None

    return Counter(vals).most_common(1)[0][0]


gt = dev.apply(majority, axis=1).values
valid = pd.notna(gt)

# ============================================================
# PRED / CONF
# ============================================================

pred = {
    k: P[k].argmax(axis=1)
    for k in P
}

conf = {
    k: P[k].max(axis=1)
    for k in P
}

# ============================================================
# BASE GOLDEN
# ============================================================

base = pred["ALL"].copy()

gold = (
    (conf["27B"] >= 0.85)
    &
    (conf["ALL"] <= 0.70)
    &
    (pred["27B"] != pred["ALL"])
)

gold_pred = base.copy()
gold_pred[gold] = pred["27B"][gold]

base_acc = np.mean(
    CLASSES[gold_pred[valid]]
    ==
    gt[valid]
)

print("=" * 80)
print("CURRENT GOLDEN")
print("=" * 80)

print(
    f"Accuracy: {base_acc:.6f}"
)

print(
    f"27B routes: {gold.sum()}"
)


# ============================================================
# 2.5B VALIDATION GATE
# ============================================================

results = []

for conf25_min in np.arange(
    0.50,
    0.96,
    0.025
):

    for require_agreement in [
        True,
        False
    ]:

        final = base.copy()

        if require_agreement:

            route = (
                gold
                &
                (pred["25B"] == pred["27B"])
                &
                (conf["25B"] >= conf25_min)
            )

        else:

            route = (
                gold
                &
                (conf["25B"] >= conf25_min)
            )

        final[route] = pred["27B"][route]

        acc = np.mean(
            CLASSES[final[valid]]
            ==
            gt[valid]
        )

        results.append({
            "conf25_min": conf25_min,
            "agreement_required": require_agreement,
            "routes": route.sum(),
            "accuracy": acc
        })


result_df = pd.DataFrame(
    results
).sort_values(
    "accuracy",
    ascending=False
)

print("\n" + "=" * 80)
print("TOP 20")
print("=" * 80)

print(
    result_df.head(20).to_string(
        index=False
    )
)


# ============================================================
# 3. 27B + 2.5B + ALL SCORE
# ============================================================

print("\n" + "=" * 80)
print("TRIPLE ENSEMBLE")
print("=" * 80)

triple_results = []

# 27B/25B 비중만 조절하고 ALL은 크게 유지
for w27 in np.arange(
    0.0,
    0.31,
    0.02
):

    for w25 in np.arange(
        0.0,
        0.51 - w27,
        0.02
    ):

        w_all = 1.0 - w27 - w25

        if w_all < 0.49:
            continue

        score = (
            w_all * P["ALL"]
            +
            w25 * P["25B"]
            +
            w27 * P["27B"]
        )

        final = score.argmax(axis=1)

        acc = np.mean(
            CLASSES[final[valid]]
            ==
            gt[valid]
        )

        triple_results.append({
            "all": w_all,
            "25B": w25,
            "27B": w27,
            "accuracy": acc
        })


triple_df = pd.DataFrame(
    triple_results
).sort_values(
    "accuracy",
    ascending=False
)

print(
    triple_df.head(20).to_string(
        index=False
    )
)


# ============================================================
# 4. 27B GOLDEN + 2.5B AGREEMENT
# ============================================================

best = result_df.iloc[0]

best_conf25 = float(
    best["conf25_min"]
)

best_agree = bool(
    best["agreement_required"]
)

print("\n" + "=" * 80)
print("BEST VALIDATION RULE")
print("=" * 80)

print(
    "25B confidence:",
    best_conf25
)

print(
    "25B agreement:",
    best_agree
)

# ============================================================
# TEST
# ============================================================

test = pd.read_csv("test.csv")

TP = {
    "ALL": softmax(
        np.load(
            "probability_all.npy"
        ).astype(np.float64)
    ),

    "25B": softmax(
        np.load(
            "probability_2.5_all.npy"
        ).astype(np.float64)
    ),

    "27B": softmax(
        np.load(
            "probability_qwen3.5_27b.npy"
        ).astype(np.float64)
    )
}

tpred = {
    k: TP[k].argmax(axis=1)
    for k in TP
}

tconf = {
    k: TP[k].max(axis=1)
    for k in TP
}

# golden
tgold = (
    (tconf["27B"] >= 0.85)
    &
    (tconf["ALL"] <= 0.70)
    &
    (tpred["27B"] != tpred["ALL"])
)

# 2.5B gate
if best_agree:

    troute = (
        tgold
        &
        (tpred["25B"] == tpred["27B"])
        &
        (tconf["25B"] >= best_conf25)
    )

else:

    troute = (
        tgold
        &
        (tconf["25B"] >= best_conf25)
    )


final_test = tpred["ALL"].copy()

final_test[troute] = tpred["27B"][
    troute
]

submission = pd.DataFrame({
    "id": test["id"],
    "answer": CLASSES[final_test]
})

submission.to_csv(
    "submission_gold_25b_validator.csv",
    index=False
)

print("\nTEST")
print(
    "Original Golden routes:",
    tgold.sum()
)

print(
    "After 25B gate:",
    troute.sum()
)

print(
    "Saved:",
    "submission_gold_25b_validator.csv"
)

CURRENT GOLDEN
Accuracy: 0.516319
27B routes: 101

TOP 20
 conf25_min  agreement_required  routes  accuracy
      0.500               False      88  0.515866
      0.825               False      30  0.515866
      0.600               False      68  0.515639
      0.625               False      65  0.515639
      0.575               False      71  0.515413
      0.500                True      49  0.515413
      0.925               False      10  0.515413
      0.750               False      44  0.515413
      0.950                True       5  0.515186
      0.525               False      78  0.515186
      0.675               False      54  0.515186
      0.900               False      15  0.515186
      0.650               False      57  0.515186
      0.750                True      27  0.515186
      0.825                True      20  0.515186
      0.850               False      23  0.515186
      0.675                True      31  0.515186
      0.950               False       5  0

In [17]:
import numpy as np
import pandas as pd
from itertools import permutations

classes = np.array(["a", "b", "c", "d"])

# 현재 argmax 기준
raw27 = np.load(
    "dev_prob_qwen3.5_27b.npy"
).astype(np.float64)

# 중요:
# 여기서는 일단 softmax 하지 말고 원본 값부터 확인
print("27B raw min/max:")
print(raw27.min(), raw27.max())

print("\n27B row sums:")
print(
    raw27.sum(axis=1)[:10]
)

gt_idx = np.array([
    {"a":0, "b":1, "c":2, "d":3}.get(x, -1)
    for x in gt
])

valid2 = gt_idx >= 0

results = []

for perm in permutations(range(4)):

    # probability column을 permute
    test = raw27[:, perm]

    pred_idx = test.argmax(axis=1)

    acc = np.mean(
        pred_idx[valid2] == gt_idx[valid2]
    )

    results.append({
        "perm": perm,
        "accuracy": acc
    })

perm_df = pd.DataFrame(results).sort_values(
    "accuracy",
    ascending=False
)

print("\n" + "="*80)
print("27B CLASS PERMUTATION TEST")
print("="*80)

print(
    perm_df.head(24).to_string(index=False)
)

27B raw min/max:
-13.25001335144043 -1.3828182090946939e-05

27B row sums:
[-12.05583769  -9.71171987 -14.35601799  -7.71519303  -5.66214657
 -11.94977546 -30.18838487 -12.32676654  -5.7176404  -28.81424727]

27B CLASS PERMUTATION TEST
        perm  accuracy
(0, 1, 2, 3)  0.446283
(0, 1, 3, 2)  0.410471
(0, 3, 1, 2)  0.328649
(0, 3, 2, 1)  0.316183
(0, 2, 1, 3)  0.310743
(3, 1, 0, 2)  0.305530
(1, 0, 2, 3)  0.300544
(3, 1, 2, 0)  0.296691
(1, 0, 3, 2)  0.264733
(0, 2, 3, 1)  0.262466
(2, 1, 0, 3)  0.249773
(3, 0, 1, 2)  0.249773
(1, 3, 0, 2)  0.238667
(3, 0, 2, 1)  0.237307
(1, 3, 2, 0)  0.229828
(1, 2, 0, 3)  0.222575
(2, 1, 3, 0)  0.205122
(2, 0, 1, 3)  0.192203
(1, 2, 3, 0)  0.177924
(3, 2, 1, 0)  0.160925
(3, 2, 0, 1)  0.157298
(2, 0, 3, 1)  0.143926
(2, 3, 1, 0)  0.121260
(2, 3, 0, 1)  0.117634


In [18]:
for name, path in {
    "ALL": "dev_prob_all.npy",
    "25B": "dev_prob_2.5_all.npy",
    "27B": "dev_prob_qwen3.5_27b.npy"
}.items():

    x = np.load(path).astype(np.float64)

    print("\n", "="*50)
    print(name)

    print("shape:", x.shape)

    print("min:", x.min())
    print("max:", x.max())

    print("mean:", x.mean())

    sums = x.sum(axis=1)

    print("row sum mean:", sums.mean())
    print("row sum min:", sums.min())
    print("row sum max:", sums.max())

    print("first 5 rows:")
    print(x[:5])


ALL
shape: (4413, 4)
min: -11.593801975250244
max: -5.1884303047700087e-05
mean: -3.059144203014205
row sum mean: -12.23657681205682
row sum min: -34.00020781005378
row sum max: -5.614826917648315
first 5 rows:
[[-0.09130709 -4.21630704 -2.7475571  -7.18505704]
 [-8.01124716 -2.26124722 -1.10499726 -0.60499724]
 [-3.42727059 -0.0522706  -4.52102053 -5.98977053]
 [-0.14412378 -2.20662379 -7.05037367 -5.70662367]
 [-4.7670536  -3.2358036  -1.76705366 -0.29830365]]

25B
shape: (4413, 4)
min: -14.437517881393433
max: -1.8000392401518184e-05
mean: -2.8986346144531163
row sum mean: -11.594538457812465
row sum min: -39.5625716445727
row sum max: -5.608771443367004
first 5 rows:
[[-0.2131572  -3.2756573  -1.99440724 -5.99440718]
 [-6.6835196  -0.83976968 -0.77726968 -2.7460196 ]
 [-1.62571222 -0.28196222 -3.34446222 -5.68821239]
 [-0.27774469 -2.24649471 -6.18399453 -2.12149471]
 [-4.7239821  -3.00523227 -1.47398227 -0.34898227]]

27B
shape: (4413, 4)
min: -13.25001335144043
max: -1.382818209

In [19]:
print("\n" + "="*80)
print("MODEL ACCURACY")
print("="*80)

for k in pred:

    acc = np.mean(
        CLASSES[pred[k][valid]]
        == gt[valid]
    )

    print(
        f"{k}: {acc:.6f}"
    )


MODEL ACCURACY
ALL: 0.514959
25B: 0.505440
27B: 0.446283


In [20]:
from sklearn.metrics import confusion_matrix

for k in pred:

    cm = confusion_matrix(
        gt[valid],
        CLASSES[pred[k][valid]],
        labels=["a","b","c","d"]
    )

    print("\n", "="*50)
    print(k)
    print(pd.DataFrame(
        cm,
        index=["GT_a","GT_b","GT_c","GT_d"],
        columns=["Pred_a","Pred_b","Pred_c","Pred_d"]
    ))


ALL
      Pred_a  Pred_b  Pred_c  Pred_d
GT_a     611     276     102      94
GT_b     186     566     238     157
GT_c     106     212     383     254
GT_d     110     132     273     712

25B
      Pred_a  Pred_b  Pred_c  Pred_d
GT_a     607     293     108      75
GT_b     210     601     223     113
GT_c     118     242     407     188
GT_d     115     192     305     615

27B
      Pred_a  Pred_b  Pred_c  Pred_d
GT_a     553     340     154      36
GT_b     191     619     310      27
GT_c      85     277     557      36
GT_d      95     277     615     240


In [21]:
# ============================================================
# 27B SELECTIVE ROUTING ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("27B SELECTIVE ROUTING")
print("=" * 80)

all_pred = pred["ALL"]
p27_pred = pred["27B"]

gt_idx = np.array([
    {"a": 0, "b": 1, "c": 2, "d": 3}.get(x, -1)
    for x in gt
])

valid_idx = gt_idx >= 0

all_correct = (
    all_pred == gt_idx
)

p27_correct = (
    p27_pred == gt_idx
)

# ALL -> 27B 변경 결과
rescue = (
    valid_idx
    & ~all_correct
    & p27_correct
)

harm = (
    valid_idx
    & all_correct
    & ~p27_correct
)

print("ALL correct :", all_correct[valid_idx].sum())
print("ALL wrong   :", (~all_correct[valid_idx]).sum())

print("27B rescue  :", rescue.sum())
print("27B harm    :", harm.sum())

print("\nNet:", rescue.sum() - harm.sum())


# ============================================================
# BY 27B PREDICTED CLASS
# ============================================================

print("\n" + "=" * 80)
print("BREAKDOWN BY 27B PREDICTED CLASS")
print("=" * 80)

for cls_idx, cls in enumerate(CLASSES):

    mask = (
        valid_idx
        & (p27_pred == cls_idx)
    )

    total = mask.sum()

    r = (
        mask
        & ~all_correct
        & p27_correct
    ).sum()

    h = (
        mask
        & all_correct
        & ~p27_correct
    ).sum()

    both_wrong = (
        mask
        & ~all_correct
        & ~p27_correct
    ).sum()

    both_correct = (
        mask
        & all_correct
        & p27_correct
    ).sum()

    print(
        f"\n27B={cls}"
    )

    print(" total       :", total)
    print(" rescue      :", r)
    print(" harm        :", h)
    print(" net         :", r - h)
    print(" both correct:", both_correct)
    print(" both wrong  :", both_wrong)

    if total > 0:
        print(
            "precision    :",
            p27_correct[mask].mean()
        )


# ============================================================
# BY ALL -> 27B TRANSITION
# ============================================================

print("\n" + "=" * 80)
print("ALL PRED -> 27B PRED")
print("=" * 80)

for a_idx, a_cls in enumerate(CLASSES):

    for b_idx, b_cls in enumerate(CLASSES):

        if a_idx == b_idx:
            continue

        mask = (
            valid_idx
            & (all_pred == a_idx)
            & (p27_pred == b_idx)
        )

        n = mask.sum()

        if n == 0:
            continue

        all_acc = all_correct[mask].mean()
        p27_acc = p27_correct[mask].mean()

        net = (
            p27_correct[mask].sum()
            - all_correct[mask].sum()
        )

        print(
            f"{a_cls} -> {b_cls}: "
            f"n={n:4d}, "
            f"ALL={all_acc:.3f}, "
            f"27B={p27_acc:.3f}, "
            f"net={net:+4d}"
        )


27B SELECTIVE ROUTING
ALL correct : 2272
ALL wrong   : 2140
27B rescue  : 551
27B harm    : 854

Net: -303

BREAKDOWN BY 27B PREDICTED CLASS

27B=a
 total       : 924
 rescue      : 72
 harm        : 85
 net         : -13
 both correct: 481
 both wrong  : 286
precision    : 0.5984848484848485

27B=b
 total       : 1513
 rescue      : 185
 harm        : 257
 net         : -72
 both correct: 434
 both wrong  : 637
precision    : 0.4091209517514871

27B=c
 total       : 1636
 rescue      : 258
 harm        : 499
 net         : -241
 both correct: 299
 both wrong  : 580
precision    : 0.3404645476772616

27B=d
 total       : 339
 rescue      : 36
 harm        : 13
 net         : 23
 both correct: 204
 both wrong  : 86
precision    : 0.7079646017699115

ALL PRED -> 27B PRED
a -> b: n= 181, ALL=0.492, 27B=0.304, net= -34
a -> c: n=  67, ALL=0.493, 27B=0.343, net= -10
a -> d: n=  30, ALL=0.267, 27B=0.533, net=  +8
b -> a: n= 110, ALL=0.500, 27B=0.400, net= -11
b -> c: n= 185, ALL=0.411, 27B=

In [22]:
import numpy as np
import pandas as pd
from itertools import product

CLASSES = np.array(["a", "b", "c", "d"])

all_pred = pred["ALL"]
p27_pred = pred["27B"]

all_conf = conf["ALL"]
p27_conf = conf["27B"]

gt_idx = np.array([
    {"a": 0, "b": 1, "c": 2, "d": 3}.get(x, -1)
    for x in gt
])

valid = gt_idx >= 0

base = all_pred.copy()

base_acc = np.mean(
    base[valid] == gt_idx[valid]
)

print("=" * 80)
print("BASE")
print("=" * 80)
print("ALL:", base_acc)


# ============================================================
# 1. 허용할 transition
# ============================================================

allowed_transitions = [
    ("a", "d"),
    ("b", "d"),
    ("c", "a"),
    ("c", "b"),
    ("c", "d"),
]

allowed_pairs = set(
    (
        np.where(CLASSES == a)[0][0],
        np.where(CLASSES == b)[0][0]
    )
    for a, b in allowed_transitions
)


# ============================================================
# 2. 공통 confidence threshold 탐색
# ============================================================

results = []

for th27 in np.arange(0.50, 0.991, 0.01):

    final = base.copy()

    route = np.zeros(len(base), dtype=bool)

    for a, b in allowed_pairs:

        route |= (
            valid
            &
            (all_pred == a)
            &
            (p27_pred == b)
            &
            (p27_conf >= th27)
        )

    final[route] = p27_pred[route]

    acc = np.mean(
        final[valid] == gt_idx[valid]
    )

    results.append({
        "threshold": th27,
        "routes": route.sum(),
        "accuracy": acc,
        "gain": acc - base_acc
    })


df = pd.DataFrame(results).sort_values(
    "accuracy",
    ascending=False
)

print("\n" + "=" * 80)
print("COMMON THRESHOLD")
print("=" * 80)

print(df.head(20).to_string(index=False))


# ============================================================
# 3. Transition별 단독 분석
# ============================================================

print("\n" + "=" * 80)
print("TRANSITION THRESHOLD ANALYSIS")
print("=" * 80)

for a_cls, b_cls in allowed_transitions:

    a = np.where(CLASSES == a_cls)[0][0]
    b = np.where(CLASSES == b_cls)[0][0]

    mask = (
        valid
        &
        (all_pred == a)
        &
        (p27_pred == b)
    )

    n = mask.sum()

    if n == 0:
        continue

    print(
        f"\n{a_cls} -> {b_cls} | n={n}"
    )

    rows = []

    for th in np.arange(0.50, 0.991, 0.025):

        sub = mask & (p27_conf >= th)

        if sub.sum() == 0:
            continue

        acc27 = np.mean(
            p27_pred[sub] == gt_idx[sub]
        )

        acc_all = np.mean(
            all_pred[sub] == gt_idx[sub]
        )

        rows.append({
            "threshold": th,
            "n": sub.sum(),
            "ALL": acc_all,
            "27B": acc27,
            "gain": acc27 - acc_all
        })

    tmp = pd.DataFrame(rows)

    print(
        tmp.sort_values(
            "gain",
            ascending=False
        ).head(5).to_string(index=False)
    )


# ============================================================
# 4. 가장 좋은 transition 조합 탐색
# ============================================================

# transition별 binary ON/OFF
transition_masks = {}

for a_cls, b_cls in allowed_transitions:

    a = np.where(CLASSES == a_cls)[0][0]
    b = np.where(CLASSES == b_cls)[0][0]

    transition_masks[(a_cls, b_cls)] = (
        valid
        &
        (all_pred == a)
        &
        (p27_pred == b)
    )


combo_results = []

for switches in product([False, True], repeat=len(allowed_transitions)):

    final = base.copy()
    route = np.zeros(len(base), dtype=bool)

    for use, pair in zip(
        switches,
        allowed_transitions
    ):
        if use:
            route |= transition_masks[pair]

    final[route] = p27_pred[route]

    acc = np.mean(
        final[valid] == gt_idx[valid]
    )

    row = {
        "accuracy": acc,
        "gain": acc - base_acc,
        "routes": route.sum(),
    }

    for use, pair in zip(
        switches,
        allowed_transitions
    ):
        row[f"{pair[0]}->{pair[1]}"] = use

    combo_results.append(row)


combo_df = pd.DataFrame(
    combo_results
).sort_values(
    "accuracy",
    ascending=False
)

print("\n" + "=" * 80)
print("BEST TRANSITION COMBINATIONS")
print("=" * 80)

print(
    combo_df.head(20).to_string(
        index=False
    )
)

BASE
ALL: 0.514959202175884

COMMON THRESHOLD
 threshold  routes  accuracy     gain
      0.50     258  0.524252 0.009293
      0.51     248  0.523345 0.008386
      0.52     237  0.522439 0.007480
      0.53     229  0.521759 0.006800
      0.56     202  0.521759 0.006800
      0.55     213  0.521532 0.006573
      0.54     223  0.521079 0.006120
      0.57     195  0.521079 0.006120
      0.58     190  0.521079 0.006120
      0.62     165  0.520852 0.005893
      0.60     179  0.520626 0.005666
      0.59     182  0.520399 0.005440
      0.61     173  0.520399 0.005440
      0.63     154  0.519946 0.004986
      0.64     150  0.519946 0.004986
      0.65     143  0.519266 0.004306
      0.67     133  0.518586 0.003626
      0.69     125  0.518586 0.003626
      0.77      89  0.518586 0.003626
      0.70     122  0.518586 0.003626

TRANSITION THRESHOLD ANALYSIS

a -> d | n=30
 threshold  n      ALL      27B     gain
     0.875  9 0.222222 0.777778 0.555556
     0.850  9 0.222222 0.777

In [23]:
# ============================================================
# TRANSITION + CONFIDENCE OPTIMIZATION
# ============================================================

from itertools import product
import numpy as np
import pandas as pd

transitions = [
    ("a", "d"),
    ("b", "d"),
    ("c", "a"),
    ("c", "b"),
    ("c", "d"),
]

# ------------------------------------------------------------
# 각 transition의 threshold 후보 생성
# ------------------------------------------------------------

candidate_thresholds = {}

for a_cls, b_cls in transitions:

    a = np.where(CLASSES == a_cls)[0][0]
    b = np.where(CLASSES == b_cls)[0][0]

    mask = (
        valid
        &
        (all_pred == a)
        &
        (p27_pred == b)
    )

    rows = []

    for th in np.arange(0.50, 0.991, 0.025):

        sub = mask & (p27_conf >= th)

        n = sub.sum()

        if n < 3:
            continue

        all_acc = np.mean(
            all_pred[sub] == gt_idx[sub]
        )

        p27_acc = np.mean(
            p27_pred[sub] == gt_idx[sub]
        )

        rows.append({
            "threshold": th,
            "n": n,
            "all_acc": all_acc,
            "p27_acc": p27_acc,
            "gain": p27_acc - all_acc,
        })

    tmp = pd.DataFrame(rows)

    # gain 기준 상위 후보
    top = tmp.sort_values(
        ["gain", "n"],
        ascending=[False, False]
    ).head(3)

    # 너무 공격적인 1~2샘플 threshold를 방지하기 위한
    # 현실적인 후보도 추가
    reasonable = tmp[
        tmp["n"] >= max(5, int(mask.sum() * 0.10))
    ].sort_values(
        "gain",
        ascending=False
    ).head(2)

    candidates = sorted(
        set(
            top["threshold"].tolist()
            +
            reasonable["threshold"].tolist()
        )
    )

    candidate_thresholds[
        (a_cls, b_cls)
    ] = candidates

    print(
        f"\n{a_cls}->{b_cls}"
    )

    print(
        tmp.sort_values(
            "gain",
            ascending=False
        ).head(8).to_string(index=False)
    )

    print(
        "Candidates:",
        candidates
    )


# ------------------------------------------------------------
# transition별 threshold 조합
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMBINING TRANSITION THRESHOLDS")
print("=" * 80)

combo_results = []

candidate_lists = [
    candidate_thresholds[t]
    for t in transitions
]

for thresholds in product(*candidate_lists):

    final = all_pred.copy()

    route = np.zeros(
        len(all_pred),
        dtype=bool
    )

    for (a_cls, b_cls), th in zip(
        transitions,
        thresholds
    ):

        a = np.where(CLASSES == a_cls)[0][0]
        b = np.where(CLASSES == b_cls)[0][0]

        route |= (
            valid
            &
            (all_pred == a)
            &
            (p27_pred == b)
            &
            (p27_conf >= th)
        )

    final[route] = p27_pred[route]

    acc = np.mean(
        final[valid] == gt_idx[valid]
    )

    row = {
        "accuracy": acc,
        "gain": acc - base_acc,
        "routes": route.sum(),
    }

    for t, th in zip(transitions, thresholds):
        row[f"{t[0]}->{t[1]}"] = th

    combo_results.append(row)


combo_df = pd.DataFrame(
    combo_results
).sort_values(
    ["accuracy", "routes"],
    ascending=[False, False]
)

print(
    combo_df.head(30).to_string(
        index=False
    )
)


a->d
 threshold  n  all_acc  p27_acc     gain
     0.875  9 0.222222 0.777778 0.555556
     0.850  9 0.222222 0.777778 0.555556
     0.800 11 0.181818 0.727273 0.545455
     0.775 11 0.181818 0.727273 0.545455
     0.825 10 0.200000 0.700000 0.500000
     0.600 20 0.250000 0.700000 0.450000
     0.700 16 0.250000 0.687500 0.437500
     0.900  7 0.285714 0.714286 0.428571
Candidates: [0.7750000000000002, 0.8500000000000003, 0.8750000000000003]

b->d
 threshold  n  all_acc  p27_acc     gain
     0.500  6 0.166667 0.666667 0.500000
     0.600  4 0.250000 0.750000 0.500000
     0.575  4 0.250000 0.750000 0.500000
     0.625  4 0.250000 0.750000 0.500000
     0.550  5 0.200000 0.600000 0.400000
     0.525  5 0.200000 0.600000 0.400000
     0.650  3 0.333333 0.666667 0.333333
     0.675  3 0.333333 0.666667 0.333333
Candidates: [0.5, 0.525, 0.5750000000000001, 0.6000000000000001]

c->a
 threshold  n  all_acc  p27_acc     gain
     0.975  5 0.000000 0.600000 0.600000
     0.575 28 0.250000 0

In [36]:
import numpy as np
import pandas as pd

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

test = pd.read_csv("test.csv")

ALL = np.load("probability_all.npy").astype(np.float64)
B27 = np.load("probability_qwen3.5_27b.npy").astype(np.float64)

PA = softmax(ALL)
P27 = softmax(B27)

all_pred = PA.argmax(axis=1)
p27_pred = P27.argmax(axis=1)

all_conf = PA.max(axis=1)
p27_conf = P27.max(axis=1)

disagree = all_pred != p27_pred

rules = {
    "A_CURRENT_GOLDEN": (
        (all_conf <= 0.70)
        & (p27_conf >= 0.85)
        & disagree
    ),

    "B_RELAX_27B": (
        (all_conf <= 0.70)
        & (p27_conf >= 0.775)
        & disagree
    ),

    "C_RELAX_BOTH": (
        (all_conf <= 0.725)
        & (p27_conf >= 0.775)
        & disagree
    ),
}

for name, route in rules.items():

    final = all_pred.copy()
    final[route] = p27_pred[route]

    submission = pd.DataFrame({
        "id": test["id"],
        "answer": np.array(["a", "b", "c", "d"])[final]
    })

    filename = f"submission_{name}.csv"
    submission.to_csv(filename, index=False)

    print(
        name,
        "| routes =", route.sum(),
        "| ratio =", route.mean(),
        "| saved =", filename
    )

A_CURRENT_GOLDEN | routes = 65 | ratio = 0.012810405991328341 | saved = submission_A_CURRENT_GOLDEN.csv
B_RELAX_27B | routes = 88 | ratio = 0.017343318880567598 | saved = submission_B_RELAX_27B.csv
C_RELAX_BOTH | routes = 90 | ratio = 0.017737485218762318 | saved = submission_C_RELAX_BOTH.csv


In [ ]:
# # ==========================================
# # [현재 로드된 모델용] dev.csv 추론 및 .npy 저장
# # ==========================================
# model.eval()
# candidate_tokens = ['a', 'b', 'c', 'd']
# candidate_ids = [processor.tokenizer.encode(t, add_special_tokens=False)[0] for t in candidate_tokens]

# ROTATIONS = 4
# BATCH_SIZE = 4
# dev_df = pd.read_csv("dev.csv") # test_df 대신 dev_df 사용
# n_samples_dev = len(dev_df)

# P_total_dev = np.zeros((n_samples_dev, 4))

# for rot in range(ROTATIONS):
#     for s in tqdm(range(0, n_samples_dev, BATCH_SIZE), desc=f"Dev Inference [TTA Rot {rot}]", unit="batch"):
#         chunk = dev_df.iloc[s:s+BATCH_SIZE]
#         texts, images, perms = [], [], []

#         for _, row in chunk.iterrows():
#             img = Image.open(row["path"]).convert("RGB")
#             q = str(row["question"])
#             base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]
#             perm = [(i + rot) % 4 for i in range(4)]
#             user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

#             messages = [
#                 {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
#                 {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
#             ]
#             texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
#             images.append(img)
#             perms.append(perm)

#         inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

#         with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
#             outputs = model(**inputs)
#             logits = outputs.logits[:, -1, :].float()
#             lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

#         for k, perm in enumerate(perms):
#             for shown_i, orig_i in enumerate(perm):
#                 P_total_dev[s+k, orig_i] += lp[k, shown_i]

#         del inputs, outputs
#         torch.cuda.empty_cache()

# P_total_dev /= ROTATIONS

# # Dev 확률값 저장 (All 모델)
# np.save("dev_prob_all.npy", P_total_dev)
# print("🎉 성공: dev_prob_all.npy 가 저장되었습니다.")

In [ ]:
# from peft import PeftModel

# # 1. 1500장 모델 가중치 덮어씌우기 (학습 불필요)
# # base_model은 이미 8B로 로드되어 있으므로, PEFT 어댑터만 1500버전으로 갈아끼웁니다.
# model = PeftModel.from_pretrained(base_model, "/content/qwen_8b_1500")
# model = model.to(device)
# model.eval()

# P_total_dev_1500 = np.zeros((n_samples_dev, 4))

# # 2. 추론 루프 (Step 1과 완전히 동일)
# for rot in range(ROTATIONS):
#     for s in tqdm(range(0, n_samples_dev, BATCH_SIZE), desc=f"Dev Inference 1500 [TTA Rot {rot}]", unit="batch"):
#         chunk = dev_df.iloc[s:s+BATCH_SIZE]
#         texts, images, perms = [], [], []

#         for _, row in chunk.iterrows():
#             img = Image.open(row["path"]).convert("RGB")
#             q = str(row["question"])
#             base_opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]
#             perm = [(i + rot) % 4 for i in range(4)]
#             user_text = build_mc_prompt(q, base_opts[perm[0]], base_opts[perm[1]], base_opts[perm[2]], base_opts[perm[3]])

#             messages = [
#                 {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
#                 {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
#             ]
#             texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
#             images.append(img)
#             perms.append(perm)

#         inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

#         with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
#             outputs = model(**inputs)
#             logits = outputs.logits[:, -1, :].float()
#             lp = torch.log_softmax(logits[:, candidate_ids], dim=-1).cpu().numpy()

#         for k, perm in enumerate(perms):
#             for shown_i, orig_i in enumerate(perm):
#                 P_total_dev_1500[s+k, orig_i] += lp[k, shown_i]

#         del inputs, outputs
#         torch.cuda.empty_cache()

# P_total_dev_1500 /= ROTATIONS

# # Dev 확률값 저장 (1500 모델)
# np.save("dev_prob_1500.npy", P_total_dev_1500)
# print("🎉 성공: dev_prob_1500.npy 가 저장되었습니다.")

In [ ]:
# import numpy as np
# import pandas as pd

# # ==========================================
# # 1. 파일 로드
# # ==========================================
# test_df = pd.read_csv("test.csv")
# p_all = np.load("probability_all.npy")     # 0.928 모델
# p_1500 = np.load("probability_1500.npy")   # 0.918 모델
# classes = np.array(['a', 'b', 'c', 'd'])

# # ==========================================
# # 2. 질문 유형(Q-Type) 분류 함수
# # ==========================================
# def classify_qtype(q):
#     q = str(q).replace(" ", "")
#     if any(w in q for w in ["몇개", "몇가지", "수량", "몇병", "총몇"]): return "count"
#     if any(w in q for w in ["재질", "소재", "무엇으로"]): return "material"
#     if any(w in q for w in ["색", "컬러"]): return "color"
#     if any(w in q for w in ["분리수거", "어떻게배출", "재활용분류"]): return "recycle"
#     return "other"

# test_df['qtype'] = test_df['question'].apply(classify_qtype)

# # ==========================================
# # 3. 마진(Margin) 및 Confidence 계산
# # ==========================================
# def calculate_margin(probs):
#     sorted_p = np.sort(probs, axis=1)
#     confidence = sorted_p[:, -1]
#     margin = sorted_p[:, -1] - sorted_p[:, -2]
#     return confidence, margin

# conf_all, margin_all = calculate_margin(p_all)
# conf_1500, margin_1500 = calculate_margin(p_1500)

# test_df['pred_all'] = classes[p_all.argmax(axis=1)]
# test_df['pred_1500'] = classes[p_1500.argmax(axis=1)]
# test_df['margin_all'] = margin_all

# # ==========================================
# # 4. 스마트 라우팅 앙상블 (우승자 아이디어 적용)
# # ==========================================
# # 기본적으로 점수가 더 높은 All 모델의 예측을 따릅니다.
# final_probs = np.copy(p_all)
# swapped_count = 0

# for i in range(len(test_df)):
#     qtype = test_df.loc[i, 'qtype']
#     m_all = margin_all[i]
#     m_1500 = margin_1500[i]

#     # 두 모델의 예측이 다르고 (Disagreement),
#     if test_df.loc[i, 'pred_all'] != test_df.loc[i, 'pred_1500']:

#         # [규칙 1] All 모델이 엄청나게 헷갈려하는 경우 (마진 10% 이하)
#         # 그런데 1500 모델은 확신하고 있다면 (마진 30% 이상) 1500의 의견을 전적으로 믿어봅니다.
#         if m_all < 0.10 and m_1500 > 0.30:
#             final_probs[i] = p_1500[i]
#             swapped_count += 1

#         # [규칙 2] 숫자 세기(count) 문제에서 의견이 갈리고, All 모델이 덜 확신하는 경우
#         # Qwen 모델 자체가 count에 약하므로 둘의 확률을 5:5 섞어서(Soft Voting) 리스크를 줄입니다.
#         elif qtype == 'count' and m_all < 0.40:
#             final_probs[i] = (p_all[i] + p_1500[i]) / 2.0
#             swapped_count += 1

#         # [규칙 3] 그 외 Q-Type에서 둘 다 마진이 낮아 헷갈릴 때는 확률을 기하평균 냅니다.
#         elif m_all < 0.20 and m_1500 < 0.20:
#             p_geo = np.sqrt(np.clip(p_all[i], 1e-12, 1) * np.clip(p_1500[i], 1e-12, 1))
#             final_probs[i] = p_geo / p_geo.sum() # 정규화
#             swapped_count += 1

# # ==========================================
# # 5. 결과 저장 및 리포트
# # ==========================================
# test_df['final_answer'] = classes[final_probs.argmax(axis=1)]

# print(f"📊 [Q-Type 분포]\n{test_df['qtype'].value_counts()}\n")
# print(f"🚨 All 모델 단독 마진 10% 이하 (초고난도 문제) : {sum(margin_all < 0.1)}개")
# print(f"✨ 1500 모델의 의견을 수용하여 예측이 변경/보정된 샘플 수 : {swapped_count}개")

# submission = test_df[['id', 'final_answer']].rename(columns={'final_answer': 'answer'})
# submission.to_csv("submission_smart_ensemble.csv", index=False)
# print("🎉 스마트 앙상블 완료! 'submission_smart_ensemble.csv'를 제출해보세요.")